# Stage 10 - Pledged-securities 1D trend status (live refresh, full universe)

**Corrected design:** a purely read-only version can go stale the same way the
credit-spread screener's already-documented gap works - if a symbol flipped
after the last fetch, the saved report keeps showing the old direction/age.
This version genuinely refreshes every symbol before reporting status.

**Second correction - a real regression found while building the Stage 11
pledged-trend-following ranking:** the original version of this cell called
`run_all_instruments(..., force_refresh=True)` with a short 390-day window,
which does a full REPLACE of each symbol's saved raw data, not an update.
Every routine live-screener run was silently throwing away notebook 12's deep
multi-year history and replacing it with just that short window - after
repeated runs, every symbol's saved history had collapsed to ~1 year, and
UP-closed-flip counts per symbol fell to 1-8, nowhere near usable for a
ranking. Fixed by switching to `incremental=True`
(`process_instrument_incremental()` in `pledged_1d_pipeline.py`): it EXTENDS
existing saved history with just the recent catch-up window instead of
replacing it outright, so routine checks no longer erode the deep history
notebook 12 built. `from_date` is no longer meaningful in this mode (each
symbol's own last-saved date drives its fetch window) - only `to_date` and
`delay_seconds` still apply.

**Rate limit hit three times in a row** even after the chunk-pacing fix in
`fetch_data.py` and a 1-second between-symbol delay - close-to-the-limit
pacing (~0.35-0.4s spacing, right at the documented 3 req/sec ceiling) isn't
leaving enough real-world margin for network jitter, and repeated failures in
the same session may also mean Kite briefly throttles further after tripping
the limit once. `delay_seconds=1.0` between symbols - real headroom, not the
minimum. If this still fails, it's worth waiting a minute or two before
retrying, in case of a lingering cooldown from the earlier attempts.

In [1]:
import sys
sys.path.append("..")

from datetime import datetime, timedelta
from src.auth import get_kite
from src.instruments import build_nse_token_lookup
from src.pledged_securities import load_pledged_universe
from src.pledged_1d_pipeline import run_all_instruments

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
universe = load_pledged_universe()
symbol_list = universe["symbol"].tolist()

# incremental=True - EXTENDS each symbol's existing saved history with just
# the recent catch-up window (per-symbol, derived from its own last-saved
# date) instead of replacing it outright. from_date is unused in this mode,
# kept only because run_all_instruments still accepts it positionally.
#
# delay_seconds bumped well past the 3 req/sec limit's minimum spacing
# (~0.33s) - three failed attempts in a row means it's worth real headroom
# over cutting it close, even though it makes the full run slower.
results = run_all_instruments(
    kite, token_lookup, symbol_list,
    from_date=None,
    to_date=datetime.now(),
    incremental=True,
    delay_seconds=1.0
)
results["status"].value_counts()

[1/154] ABBOTINDIA — processing...
Fetching 2025-07-03 to 2026-08-02...
Fetching 2026-08-03 to 2026-08-07...
Saved 2870 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ABBOTINDIA_day.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/pledged_1d/ABBOTINDIA_full_report.xlsx
[1/154] ABBOTINDIA — done
[2/154] ACC — processing...
Fetching 2025-07-03 to 2026-08-02...
Fetching 2026-08-03 to 2026-08-07...
Saved 2870 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ACC_day.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/pledged_1d/ACC_full_report.xlsx
[2/154] ACC — done
[3/154] AGARWALEYE — processing...
Fetching 2025-07-03 to 2026-08-02...
Fetching 2026-08-03 to 2026-08-07...
Saved 372 candles to /Use

status
SUCCESS    140
FAILED      14
Name: count, dtype: int64

In [2]:
from src.pledged_1d_screener import check_trend_status
import pandas as pd

results = [check_trend_status(s, universe_df=universe) for s in symbol_list]
results = [r for r in results if r is not None]
status = pd.DataFrame(results).sort_values("trend_age_days")
print(f"{len(status)} of {len(symbol_list)} instruments have a current status")

No report found for BIRET - refresh it first
No report found for CMRGREEN - refresh it first
No report found for CORDELIA - refresh it first
No report found for EMBASSY - refresh it first
No report found for INDIGRID - refresh it first
No report found for IRBINVIT - refresh it first
No report found for KNACK - refresh it first
No report found for KRT - refresh it first
No report found for MINDSPACE - refresh it first
No report found for NSDL - refresh it first
No report found for NXST - refresh it first
No report found for PGINVIT - refresh it first
No report found for VAML - refresh it first
No report found for LIQUIDCASE - refresh it first
140 of 154 instruments have a current status


In [3]:
# Merge in score_for_pledged_trend_following()'s ranking (Stage 10/11) -
# answers "which of several same-day UP flips should I prioritize" instead
# of leaving it a guess. Run notebook 18 first if these files don't exist
# yet or look stale.
ranking = pd.read_excel("../exports/RANKING_pledged_trend_following.xlsx")
watchlist = pd.read_excel("../exports/WATCHLIST_pledged_trend_following.xlsx")

score_cols = ["symbol", "pledged_trend_score", "outlier_reliance_flag", "gap_risk_flag"]
ranking_slim = ranking[score_cols]

# Watchlist symbols have no score yet (below the 15 UP-closed-flip floor) -
# kept as NaN/blank rather than 0, so "no score" is never confused with
# "scored low". thin_sample flags which case applies.
watchlist_slim = watchlist[["symbol"]].copy()
for col in ["pledged_trend_score", "outlier_reliance_flag", "gap_risk_flag"]:
    watchlist_slim[col] = None
watchlist_slim["thin_sample"] = True

score_lookup = pd.concat([
    ranking_slim.assign(thin_sample=False),
    watchlist_slim
], ignore_index=True)

status = status.merge(score_lookup, on="symbol", how="left")
print(f"Scored: {status['pledged_trend_score'].notna().sum()} of {len(status)} — rest are thin-sample (< 15 UP-closed flips)")

Scored: 49 of 140 — rest are thin-sample (< 15 UP-closed flips)


## Shares

In [4]:
shares_status = status[status["security_type"] == "Equity"].reset_index(drop=True)
shares_status

,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,PGHL,DOWN,1,50.0,False,False,False,Equity,,False,39.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.58pp),NaN,False
1,IRCTC,UP,1,41.0,False,False,False,Equity,,False,None,None,None,True
2,KOTAKBANK,UP,2,45.0,False,False,False,Equity,,True,None,None,None,True
3,TATASTEEL,UP,2,43.5,False,False,False,Equity,,True,76.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.92pp),NaN,False
4,ITC,UP,3,53.5,False,False,False,Equity,,True,None,None,None,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,ASIANPAINT,UP,119,47.0,True,True,False,Equity,,True,None,None,None,True
99,RAINBOW,UP,122,71.0,True,True,False,Equity,,False,None,None,None,True
100,ANANDRATHI,UP,127,39.0,True,True,True,Equity,,False,None,None,None,True
101,BOSCHLTD,UP,127,45.5,True,True,False,Equity,,True,None,None,None,True


## ETFs

In [5]:
etf_status = status[status["security_type"] == "ETF"].reset_index(drop=True)
etf_status

,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,MON100,UP,2,57.0,False,False,False,ETF,Nasdaq / International,False,79.7,NaN,⚠ HIGH GAP RISK (>10.0%),False
1,MAFANG,UP,3,32.5,False,False,False,ETF,Nasdaq / International,False,79.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.01pp),⚠ HIGH GAP RISK (>10.0%),False
2,GROWWRAIL,UP,4,57.0,False,False,False,ETF,Railways,False,None,None,None,True
3,DIVIDEND,UP,8,53.5,False,False,False,ETF,Dividend,False,None,None,None,True
4,QUAL30IETF,UP,9,52.5,False,False,False,ETF,Quality factor,False,None,None,None,True
5,ITBEES,UP,11,64.0,False,False,False,ETF,IT / Technology,False,20.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.49pp),⚠ HIGH GAP RISK (>10.0%),False
6,PSUBNKBEES,DOWN,30,42.0,False,False,False,ETF,PSU Bank,False,36.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.18pp),⚠ HIGH GAP RISK (>10.0%),False
7,NIF100BEES,UP,32,53.5,False,False,False,ETF,Nifty 100,False,38.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),False
8,TOP100CASE,UP,32,144.5,False,False,False,ETF,Multicap / Total market,False,None,None,None,True
9,METALIETF,DOWN,45,51.0,False,False,False,ETF,Metal,False,None,None,None,True


## Top 25 UP-trending, ranked by pledged_trend_score (best first)

Sorted by score, not age — a fresh flip isn't automatically the best pick, and this is the view meant to answer that question directly when several instruments flip UP the same day. `trend_age_days` is still shown for context. Thin-sample names (score is blank) are included but sort to the bottom, not silently dropped.

### Shares

In [6]:
top25_shares_up = shares_status[shares_status["direction"] == "UP"].sort_values(
    "pledged_trend_score", ascending=False, na_position="last"
).head(25)
top25_shares_up

,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
60,TITAN,UP,52,41.0,True,False,False,Equity,,True,83.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.74pp),NaN,False
63,SBIN,UP,56,39.5,True,False,False,Equity,,True,83.6,NaN,NaN,False
41,NESTLEIND,UP,32,47.5,False,False,False,Equity,,True,83.1,NaN,NaN,False
10,MARUTI,UP,7,39.0,False,False,False,Equity,,True,82.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.46pp),NaN,False
3,TATASTEEL,UP,2,43.5,False,False,False,Equity,,True,76.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.92pp),NaN,False
81,TORNTPHARM,UP,91,57.5,True,False,False,Equity,,True,74.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.85pp),NaN,False
88,PIDILITIND,UP,108,46.5,True,True,False,Equity,,True,69.3,NaN,NaN,False
45,ULTRACEMCO,UP,36,44.5,False,False,False,Equity,,True,66.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.89pp),NaN,False
56,SHREECEM,UP,50,45.0,True,False,False,Equity,,True,60.1,NaN,NaN,False
61,SBICARD,UP,52,50.0,True,False,False,Equity,,True,59.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.70pp),NaN,False


### ETFs

In [7]:
top25_etf_up = etf_status[etf_status["direction"] == "UP"].sort_values(
    "pledged_trend_score", ascending=False, na_position="last"
).head(25)
top25_etf_up

,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,MON100,UP,2,57.0,False,False,False,ETF,Nasdaq / International,False,79.7,NaN,⚠ HIGH GAP RISK (>10.0%),False
1,MAFANG,UP,3,32.5,False,False,False,ETF,Nasdaq / International,False,79.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.01pp),⚠ HIGH GAP RISK (>10.0%),False
34,PVTBANIETF,UP,121,67.0,True,False,False,ETF,Private Bank,False,68.9,NaN,⚠ HIGH GAP RISK (>10.0%),False
28,CONSUMBEES,UP,114,91.0,True,False,False,ETF,FMCG / Consumption,False,64.7,NaN,⚠ HIGH GAP RISK (>10.0%),False
14,BANKBEES,UP,56,43.0,True,False,False,ETF,Bank Nifty / Banking,False,62.2,NaN,⚠ HIGH GAP RISK (>10.0%),False
33,JUNIORBEES,UP,119,61.5,True,True,False,ETF,Nifty Next 50,False,50.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.60pp),⚠ HIGH GAP RISK (>10.0%),False
10,INFRABEES,UP,51,55.5,False,False,False,ETF,Infra,False,42.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),False
11,NIFTYBEES,UP,51,55.5,False,False,False,ETF,Nifty 50 / Broad Market,False,40.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.75pp),⚠ HIGH GAP RISK (>10.0%),False
7,NIF100BEES,UP,32,53.5,False,False,False,ETF,Nifty 100,False,38.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),False
5,ITBEES,UP,11,64.0,False,False,False,ETF,IT / Technology,False,20.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.49pp),⚠ HIGH GAP RISK (>10.0%),False


## Today's fresh UP flips, ranked by score

The direct answer to "6 instruments flipped UP the same day, which do I buy" — filtered to only flips that happened in the most recent session (`trend_age_days <= 1`), sorted by `pledged_trend_score`. Empty is a normal, common result (this signal doesn't fire on most stocks most days) — it means nothing new to act on right now, not that something's broken.

In [8]:
fresh_up = status[(status["direction"] == "UP") & (status["trend_age_days"] <= 1)].sort_values(
    "pledged_trend_score", ascending=False, na_position="last"
)[["symbol", "security_type", "trend_age_days", "pledged_trend_score",
   "outlier_reliance_flag", "gap_risk_flag", "thin_sample", "etf_category"]]
print(f"{len(fresh_up)} fresh UP flip(s) today")
fresh_up

1 fresh UP flip(s) today


,symbol,security_type,trend_age_days,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample,etf_category
1,IRCTC,Equity,1,None,None,None,True,
